# 15장. 하나의 데이터 분석 프로젝트로 완성하기

이 노트북은 **필수 분석 Gate와 선택 단계를 분리**하고, 마지막에 Validation·Manifest·Submission Gate를 함께 확인합니다.

금액성 EDA의 `total_sales` 컬럼은 이 프로젝트에서 `completed` 주문의 `quantity × unit_price` 합계, 즉 **완료 주문 기준 금액**을 뜻합니다. 회계상 순매출로 단정하지 않습니다.

기본 실행은 외부 네트워크를 호출하지 않습니다.


## 학습 목표

- PK/FK·병합·완료 주문 총합을 필수 Gate로 확인합니다.
- 분류·외부 데이터·LLM을 선택 단계로 구분합니다.
- 고객 공개 결과에서 원본 식별정보를 제거합니다.
- 외부 데이터의 provenance와 날짜 coverage를 확인합니다.
- 빈 LLM 로그와 실제 사용 기록을 구분합니다.
- `PASS/WARN/FAIL`과 `READY/READY_WITH_WARNINGS/BLOCKED`를 구분합니다.


## 1. 프로젝트 루트 설정


In [ ]:
from pathlib import Path
import sys

def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트 루트:', PROJECT_ROOT)


## 2. 전체 파이프라인 실행

원본 데이터가 없다면 먼저 프로젝트 루트 터미널에서 `python scripts/generate_sample_data.py`를 실행합니다.


In [ ]:
from src.final_project import run_final_project

result = run_final_project(PROJECT_ROOT, random_state=42)
display(result['submission_status'])


`BLOCKED`이면 최종 제출 상태가 아닙니다. 산출물이 생성되어 있어도 `FAIL` 또는 필수 파일 누락을 먼저 수정합니다.


## 3. 데이터 구조·PK/FK Gate


In [ ]:
core = result['core']
display(core['dataset_summary'])
display(core['preprocessing_comparison'])
display(core['key_duplicate_checks'])
display(core['relationship_checks'])
display(core['public_tables']['merge_checks'])


기본 키는 `customer_id`, `product_id`, `order_id`, `order_item_id`를 확인합니다. 병합은 `validate`, 행 수 보존, 미매칭을 함께 확인합니다.


## 4. 완료 주문 범위와 다섯 총합 검증


In [ ]:
public_tables = core['public_tables']
display(public_tables['amount_scope_summary'])
display(public_tables['total_consistency_check'])
display(public_tables['core_validation'])


다음 값은 같은 `completed` 범위를 사용하므로 일치해야 합니다.

```text
completed_total = category_total = monthly_total = customer_total = product_total
```


## 5. EDA와 공개 결과 개인정보 확인


In [ ]:
display(public_tables['category_sales'].head(10))
display(public_tables['monthly_sales'].head(12))
display(public_tables['customer_sales'].head(10))
display(public_tables['product_sales'].head(10))
display(public_tables['order_status_summary'])

private_columns = {'customer_id', 'name', 'email', 'phone', 'address'}
print('고객 공개 결과 민감 컬럼 교집합:', private_columns & set(public_tables['customer_sales'].columns))


## 6. 주문 취소 분류 — 선택 단계


In [ ]:
classification_result = result['classification']
display(classification_result['status'])
display(classification_result['validation_comparison'])
display(classification_result['test_metrics'])
display(classification_result['confusion_matrix'])


`completed=0`, `cancelled=1`만 사용합니다. 데이터 부족이면 `skipped`, 다른 선택 단계 계약 문제면 `warning`으로 기록합니다. 모델·임계값은 validation에서 선택하고 test는 고정 후 최종 평가에만 사용합니다.


## 7. 외부 데이터 — 선택 단계


In [ ]:
external_result = result['external']
display(external_result['status'])
display(external_result.get('quality_checks'))
display(external_result['comparison'])
display(external_result['merge_check'])


실제 통합 파일은 `data/external/processed/holidays.csv`이며 `date, holiday_name, is_holiday`뿐 아니라 `provider, source_url, data_reference_date, license_or_terms` provenance가 필요합니다. 미매칭 내부 날짜를 자동으로 일반일로 간주하지 않습니다.


## 8. LLM 사용 Evidence


In [ ]:
display(result['llm_usage_log'])
display(result['llm_usage_validation'])


LLM 로그 파일이 이미 있으면 파이프라인은 실제 사용 기록을 덮어쓰지 않습니다. 빈 템플릿은 `execution_status=not_executed`이며 사용 증거가 아닙니다.


## 9. 프로젝트 Validation


In [ ]:
validation = result['validation']
display(validation)
display(validation['status'].value_counts())


- `PASS`: 기준 충족
- `WARN`: 선택 단계 미실행 또는 해석 제한
- `FAIL`: 제출 전 반드시 수정


## 10. Manifest와 Submission Gate


In [ ]:
manifest = result['manifest']
display(manifest)
display(result['submission_status'])

required_missing = manifest.loc[manifest['required'] & (~manifest['exists'] | ~manifest['nonempty'])]
display(required_missing)


SHA-256은 파일 변경 탐지 Evidence입니다. 분석 내용의 타당성은 Validation이 담당합니다. `BLOCKED`이면 보고서와 파일이 존재해도 제출 완료가 아닙니다.


## 11. 최종 산출물 확인


In [ ]:
for name, path in result['output_paths'].items():
    print(name, 'OK' if path.exists() and path.stat().st_size > 0 else 'MISSING/EMPTY', path)

print('최종 보고서:', result['final_report_path'])
print('Submission status:', result['submission_status_path'])
print('Manifest:', result['deliverables_path'])


## 12. 재실행과 과정 마무리

프로젝트 루트 터미널에서 같은 흐름을 재실행할 수 있습니다.

```powershell
python scripts/generate_sample_data.py
python scripts/run_final_project.py
```

최종 체크 순서는 **Question → Data → Understand → Preprocess → Validate → EDA → Visualize → Model when justified → Use LLM carefully → External data when justified → Design automation → Validate again → Report → Manifest → Submission Gate → Reproduce** 입니다.
